In [ ]:
import customtkinter as ctk 
from tkinter import messagebox, filedialog
import tkinter as tk
import sqlite3
import google.generativeai as genai
import smtplib
from email.message import EmailMessage
import threading 

# ReportLab Imports
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.graphics.shapes import Drawing 
from reportlab.graphics.charts.barcharts import VerticalBarChart 
from reportlab.lib.units import inch

# --- CONFIGURATION ---
API_KEY = "---MY API KEY---" # Use your real key (NOTE: This key is non-functional and should be replaced)

# Configure Google Gemini
genai.configure(api_key=API_KEY)

# Database Setup
db_file = "patient_data_v2.db"

def init_db():
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT, age TEXT, gender TEXT, 
            symptoms TEXT, diagnosis TEXT, date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()

# --- MODERN UI SETUP ---
ctk.set_appearance_mode("Dark") 
ctk.set_default_color_theme("blue") 

# ==============================================================================
# PRODOC APPLICATION CLASS
# ==============================================================================

class ProDocApp(ctk.CTk):
    def __init__(self):
        super().__init__()

        # Window Config
        self.title("ProDoc AI | Advanced Medical Assistant")
        self.geometry("1100x700")

        # Layout Configuration
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # --- LEFT SIDEBAR (Controls) ---
        self.sidebar_frame = ctk.CTkFrame(self, width=200, corner_radius=0)
        self.sidebar_frame.grid(row=0, column=0, sticky="nsew")
        self.sidebar_frame.grid_rowconfigure(4, weight=1)

        self.logo_label = ctk.CTkLabel(self.sidebar_frame, text="ProDoc AI ✚", font=ctk.CTkFont(size=24, weight="bold"))
        self.logo_label.grid(row=0, column=0, padx=20, pady=(20, 10))

        self.btn_predict = ctk.CTkButton(self.sidebar_frame, text="Analyze & Predict", command=self.start_prediction_thread, fg_color="#2CC985", text_color="black")
        self.btn_predict.grid(row=1, column=0, padx=20, pady=10)

        self.btn_pdf = ctk.CTkButton(self.sidebar_frame, text="Download Report (PDF)", command=self.generate_report)
        self.btn_pdf.grid(row=2, column=0, padx=20, pady=10)

        self.btn_email = ctk.CTkButton(self.sidebar_frame, text="Email Report", command=self.send_email_popup)
        self.btn_email.grid(row=3, column=0, padx=20, pady=10)

        self.status_label = ctk.CTkLabel(self.sidebar_frame, text="Status: Ready", text_color="gray")
        self.status_label.grid(row=5, column=0, padx=20, pady=20)

        # --- MAIN AREA (Inputs) ---
        self.scrollable_frame = ctk.CTkScrollableFrame(self, label_text="Patient Vitals & Symptoms")
        self.scrollable_frame.grid(row=0, column=1, sticky="nsew", padx=20, pady=20)
        
        # Input Dictionary to store widgets
        self.entries = {}
        
        self.fields = [
            ("Patient Name", "John Doe"), ("Age", "30"), ("Gender", "Male"), ("Weight (kg)", "75"),
            ("Blood Pressure", "120/80"), ("Sugar Level", "90 mg/dL"), ("Temperature (°F)", "98.6"),
            ("Oxygen Level (%)", "98"), ("Pulse Rate", "72"), ("Breath Rate", "16"),
            ("Suffering Since", "2 days"), ("Chronic Diseases", "None")
        ]

        # Create Grid of Inputs
        for i, (label_text, placeholder) in enumerate(self.fields):
            row = i // 2
            col = (i % 2) * 2
            
            lbl = ctk.CTkLabel(self.scrollable_frame, text=label_text, anchor="w")
            lbl.grid(row=row, column=col, padx=10, pady=(10, 0), sticky="w")
            
            entry = ctk.CTkEntry(self.scrollable_frame, placeholder_text=placeholder, width=250)
            entry.grid(row=row, column=col+1, padx=10, pady=(0, 10))
            self.entries[label_text] = entry

        # Large Text Box for Symptoms
        self.lbl_symp = ctk.CTkLabel(self.scrollable_frame, text="Detailed Symptoms", anchor="w")
        self.lbl_symp.grid(row=len(self.fields)//2 + 1, column=0, padx=10, pady=(20, 0), sticky="w")
        
        self.txt_symptoms = ctk.CTkTextbox(self.scrollable_frame, height=100, width=600)
        self.txt_symptoms.grid(row=len(self.fields)//2 + 2, column=0, columnspan=4, padx=10, pady=(5, 20))

        # --- RIGHT/BOTTOM AREA (Results) ---
        self.result_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.result_frame.grid(row=1, column=1, sticky="nsew", padx=20, pady=(0, 20))
        
        self.lbl_result = ctk.CTkLabel(self.result_frame, text="AI Diagnosis Results:", font=ctk.CTkFont(size=16, weight="bold"))
        self.lbl_result.pack(anchor="w")
        
        self.result_box = ctk.CTkTextbox(self.result_frame, height=150, font=ctk.CTkFont(size=14))
        self.result_box.pack(fill="both", expand=True)

    def get_vitals(self):
        data = {k: v.get() for k, v in self.entries.items()}
        data["Symptoms"] = self.txt_symptoms.get("1.0", "end-1c")
        return data

    def start_prediction_thread(self):
        self.status_label.configure(text="Status: Analyzing...", text_color="#FFD700")
        self.result_box.delete("1.0", "end")
        self.result_box.insert("1.0", "Consulting AI Doctor... Please wait.")
        threading.Thread(target=self.predict_disease).start()

    def predict_disease(self):
        try:
            vitals = self.get_vitals()
            
            # --- STRUCTURED PROMPT FOR CLINICAL OUTPUT ---
            prompt = (
                f"Act as a professional medical consultant and strictly provide your response using the following headings.\n"
                f"Analyze these patient vitals carefully: {vitals}\n\n"
                f"--- OUTPUT HEADINGS ---\n"
                f"Clinical Impression:\n"
                f"Possible Etiology:\n"
                f"Provisional Diagnosis:\n"
                f"Recommended Investigations:\n"
                f"Immediate Management Plan:\n"
                f"Dietary Advice:\n"
                f"Precautions:\n"
                f"Prognosis:\n"
                f"Notes:\n\n"
                f"Format each section with concise bullet points. Start immediately with the first heading."
            )

            # --- GOOGLE GEMINI CALL ---
            model = genai.GenerativeModel('gemini-2.5-flash')
            response = model.generate_content(prompt)
            diagnosis = response.text

            # Update UI 
            self.result_box.delete("1.0", "end")
            self.result_box.insert("1.0", diagnosis)
            self.status_label.configure(text="Status: Complete", text_color="#2CC985")
            
            # Save to DB (Simplified)
            self.save_to_db(vitals["Patient Name"], vitals["Age"], vitals["Gender"], vitals["Symptoms"], diagnosis)

        except Exception as e:
            self.result_box.delete("1.0", "end")
            self.result_box.insert("1.0", f"Error: {str(e)}")
            self.status_label.configure(text="Status: Error", text_color="red")

    def save_to_db(self, name, age, gender, symptoms, diagnosis):
        conn = sqlite3.connect(db_file)
        c = conn.cursor()
        c.execute("INSERT INTO patients (name, age, gender, symptoms, diagnosis) VALUES (?, ?, ?, ?, ?)",
                  (name, age, gender, symptoms, diagnosis))
        conn.commit()
        conn.close()

    # --- HELPER FUNCTION TO PARSE AND ADD STRUCTURED TEXT TO PDF ---
    def add_structured_text(self, flowables, report_text):
        styles = getSampleStyleSheet()
        
        # Define a specific style for section titles
        section_style = ParagraphStyle('SectionStyle', parent=styles['Heading2'], 
                                         fontSize=14, spaceBefore=15, spaceAfter=5, 
                                         textColor=colors.HexColor('#003366'))
        
        # Define a list style for bullet points
        list_style = ParagraphStyle('ListStyle', parent=styles['Normal'], 
                                         fontSize=10, leftIndent=20, spaceBefore=3, 
                                         bulletText='•')

        # Split the text by the main headings defined in the prompt
        sections = report_text.split('\n\n')
        
        # Process each section
        for section in sections:
            if ':' in section:
                # Assuming the first line is the heading
                heading, content = section.split(':', 1)
                
                # Add the Heading to the flowables
                flowables.append(Paragraph(f"<b>{heading.strip()}:</b>", section_style))
                
                # Split the content into bullet points
                points = [p.strip() for p in content.split('*') if p.strip()]
                
                if not points:
                    # Fallback for plain text content if no bullets were used
                    points = [c.strip() for c in content.split('\n') if c.strip()]
                
                # Add each point as a separate paragraph with a bullet
                for point in points:
                    flowables.append(Paragraph(point, list_style))
                
                flowables.append(Spacer(1, 0.1 * inch))

    # --- ADVANCED REPORTLAB PDF GENERATION (FIXED AGAIN) ---
    def generate_report(self):
        report_text = self.result_box.get("1.0", "end-1c")
        if len(report_text) < 10:
            messagebox.showwarning("Empty", "Please generate a diagnosis first.")
            return

        pdf_file = filedialog.asksaveasfilename(defaultextension=".pdf", filetypes=[("PDF Files", "*.pdf")])
        if not pdf_file: return

        try:
            doc = SimpleDocTemplate(pdf_file, pagesize=letter,
                                     rightMargin=inch/2, leftMargin=inch/2,
                                     topMargin=inch/2, bottomMargin=inch/2)
            
            styles = getSampleStyleSheet()
            flowables = []
            vitals = self.get_vitals()
            
            # --- 1. HEADER SECTION ---
            header_style = ParagraphStyle('HeaderStyle', parent=styles['Heading1'], 
                                         fontSize=24, alignment=1, spaceAfter=5, 
                                         textColor=colors.HexColor('#003366'))
            sub_header_style = ParagraphStyle('SubHeaderStyle', parent=styles['Normal'], 
                                         fontSize=10, alignment=1, spaceAfter=15)
            
            flowables.append(Paragraph("<b>PRODOC AI CLINICAL DIAGNOSIS REPORT</b>", header_style))
            flowables.append(Paragraph("Generated by Virtual Medical Assistant - Consult a Physician for Confirmation", sub_header_style))
            flowables.append(Paragraph("<hr/>", styles['Normal'])) # Separator Line
            flowables.append(Spacer(1, 0.2 * inch))

            # --- 2. PATIENT DEMOGRAPHICS & VITALS SUMMARY ---
            flowables.append(Paragraph("<b>Patient Summary</b>", styles['Heading2']))
            
            patient_data = [
                ["Name:", vitals['Patient Name'], "Age:", vitals['Age'], "Gender:", vitals['Gender']],
                ["Weight (kg):", vitals['Weight (kg)'], "Suffering Since:", vitals['Suffering Since'], "Chronic Disease:", vitals['Chronic Diseases']],
            ]

            # Custom Table Style for Info Boxes
            table_style = TableStyle([
                ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#E0E0E0')),
                ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
                ('FONTNAME', (0, 1), (-1, 1), 'Helvetica-Bold', 10, 0), # Bold labels
                ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
                ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
                ('LEFTPADDING', (0, 0), (-1, -1), 6),
                ('RIGHTPADDING', (0, 0), (-1, -1), 6),
            ])

            table = Table(patient_data, colWidths=['*', '*', '*', '*', '*', 0.87*inch])
            table.setStyle(table_style)
            flowables.append(table)
            flowables.append(Spacer(1, 0.1 * inch))

            # Summary Vitals Line
            vitals_summary = f"<b>Current Vitals:</b> BP: {vitals['Blood Pressure']} | Pulse: {vitals['Pulse Rate']} | Temp: {vitals['Temperature (°F)']} | Oxygen: {vitals['Oxygen Level (%)']}"
            flowables.append(Paragraph(vitals_summary, styles['Normal']))
            flowables.append(Spacer(1, 0.4 * inch))
            
            # --- 3. STRUCTURED AI DIAGNOSIS ---
            flowables.append(Paragraph("<b>CLINICAL FINDINGS AND MANAGEMENT</b>", styles['Heading1']))
            flowables.append(Spacer(1, 0.2 * inch))

            # Use the helper function to parse and add structured text
            self.add_structured_text(flowables, report_text)
            
            flowables.append(Spacer(1, 0.4 * inch))
            
            # --- 4. DATA VISUALIZATION (CHART) ---
            flowables.append(Paragraph("<b>Key Vitals Comparison Chart</b>", styles['Heading2']))
            
            try:
                pulse = float(vitals['Pulse Rate'])
                # Assuming BP is something like '120/80', we'll only chart the top number for comparison
                bp_systolic = float(vitals['Blood Pressure'].split('/')[0])
            except ValueError:
                pulse = 70.0 
                bp_systolic = 120.0 

            # Data Structure: [Series 1 (Patient), Series 2 (Ideal), Series 3 (Upper)]
            chart_data = [
                [pulse, bp_systolic],         # Series 1: Patient Values (Index 0)
                [72, 120],                    # Series 2: Ideal Values (Index 1)
                [85, 140],                    # Series 3: Upper Limit (Index 2)
            ] 

            drawing = Drawing(450, 200) 
            
            bc = VerticalBarChart()
            bc.x = 50
            bc.y = 50
            bc.height = 125
            bc.width = 350 
            bc.data = chart_data
            
            # CRITICAL FIX: Direct color assignment using the bars list property (Most robust method)
            bar_colors = [colors.blue, colors.red, colors.green]
            
            for i, color in enumerate(bar_colors):
                if i < len(bc.bars):
                    bc.bars[i].fillColor = color
            
            # Styling the bars
            bc.groupSpacing = 10
            bc.barSpacing = 2.5
            bc.barWidth = 10
            
            bc.valueAxis.valueMin = 0
            bc.valueAxis.valueMax = 150 
            bc.valueAxis.valueStep = 30
            
            bc.categoryAxis.labels.boxAnchor = 'n'
            bc.categoryAxis.categoryNames = ['Pulse (bpm)', 'BP Sys (mmHg)']
            
            drawing.add(bc)
            
            # Add Legend
            legend_text = "<b>Legend:</b> Patient Value ($\color{blue}{Blue}$) | Ideal Value ($\color{red}{Red}$) | Upper Limit ($\color{green}{Green}$)"
            flowables.append(Paragraph(legend_text, styles['Normal']))
            
            flowables.append(drawing)
            
            # --- BUILD PDF ---
            doc.build(flowables)
            messagebox.showinfo("Success", f"Advanced Report generated successfully and saved to: {pdf_file}")

        except Exception as e:
            messagebox.showerror("ReportLab Error", f"Failed to generate advanced PDF Report. Error: {e}")


    def send_email_popup(self):
        # A simple input dialog for email
        dialog = ctk.CTkInputDialog(text="Enter Recipient Email:", title="Email Report")
        email = dialog.get_input()
        if email:
            # Note: This will need a functional App Password for the SENDER account.
            self.send_email(email)

    def send_email(self, recipient):
        # NOTE: You still need an App Password for Gmail
        SENDER = "---MY MAIL ID---"
        PASSWORD = "YOUR_APP_PASSWORD_HERE"  # Replace with actual App Password
        
        msg = EmailMessage()
        msg.set_content(self.result_box.get("1.0", "end-1c"))
        msg['Subject'] = "ProDoc AI Report"
        msg['From'] = SENDER
        msg['To'] = recipient

        try:
            # Corrected SMTP server address
            with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
                server.login(SENDER, PASSWORD)
                server.send_message(msg)
            messagebox.showinfo("Success", "Email sent!")
        except Exception as e:
            messagebox.showerror("Error", f"Could not send email. Did you use an App Password?\nError: {e}")

if __name__ == "__main__":
    app = ProDocApp()
    app.mainloop()